# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")

from finintel.agent.graph import AgentPipeline

agent = AgentPipeline()
print(f"Model: {agent.model}")
print(f"Per-query K: {agent.per_query_k}")
print(f"Max context chunks: {agent.max_context_chunks}")

# The multi-company question that broke baseline

In [ ]:
result = agent.answer(
    "How do Google and Microsoft each frame competitive risks in their cloud businesses?"
)

print("=" * 70)
print("PLANNER OUTPUT")
print("=" * 70)
for sq in result.sub_queries:
    print(f"  • [{sq.get('ticker') or 'any'}/{sq.get('section') or 'any'}] {sq['question']}")

print("\n" + "=" * 70)
print("ANSWER")
print("=" * 70)
print(result.answer)

print("\n" + "=" * 70)
print(f"SOURCES ({len(result.sources)})")
print("=" * 70)
from collections import Counter
print("By ticker:", dict(Counter(s["ticker"] for s in result.sources)))
print("By section:", dict(Counter(s["section"] for s in result.sources)))
for s in result.sources:
    print(f"  - {s['ticker']:6} {s['section']:14} {s['chunk_id']}")

print(f"\nTokens: {result.input_tokens:,} in / {result.output_tokens:,} out")

# Compare against the baseline single-shot

In [ ]:
from finintel.agent.generator import RAGPipeline

baseline = RAGPipeline()
baseline_result = baseline.answer(
    "How do Google and Microsoft each frame competitive risks in their cloud businesses?"
)

print("BASELINE source distribution:")
print(dict(Counter(s["ticker"] for s in baseline_result.sources)))

print("\nAGENT source distribution:")
print(dict(Counter(s["ticker"] for s in result.sources)))